In [1]:
import zipfile
import scipy
import numpy as np
import pandas as pd

from code_utils.utils_clustering import clustering_metric_search, cluster_results
from code_utils.utils_basic import PROJECT_PATH

In [2]:
def load_csv_from_zip(zip_path, csv_path, **kwargs):
    """
    Load a CSV file from a ZIP archive into a pandas DataFrame.
    """
    with zipfile.ZipFile(zip_path, 'r') as zf:
        with zf.open(csv_path) as f:
            df = pd.read_csv(f, **kwargs)
    return df

# Clustering temporal bus stop volume

#### (1) Data preparation

In [3]:
# Load data
zip_path = PROJECT_PATH / 'data/features.zip'

# Load edge list for space L
volume = load_csv_from_zip(
    zip_path, csv_path = 'busstop_volume_workday.csv',
    header = 0, index_col = False,
    dtype = {'pt_code': str})

# Time columns, from 6:00 to 23:00
time_cols = [str(t).zfill(2) for t in range(6, 24)]

# Input data
data = volume[time_cols].copy()
# Remove bus stop with low volume (total tap-in volume < 100)
data = data[data.sum(axis=1) >= 100.].copy()
# Convert to percentage of total volume
data = data.div(data.sum(axis=1), axis=0)

print('Data shape:', data.shape)

Data shape: (4826, 18)


#### (2) Parameter search for clustering metrics

In [4]:
# plot_chi_score(data, max_clusters=15)
cluster_metrics = {}

# Parameter search for clustering metrics
#  cluster number: 2 to 15
#  clustering method: kmeans, hierarchical (ward, complete, average, single)
for _label, _params in zip(
    ['kmeans', 'hierarchical_ward', 'hierarchical_complete', 'hierarchical_average', 'hierarchical_single'],
    [('kmeans', None), ('hierarchical', 'ward'), ('hierarchical', 'complete'), ('hierarchical', 'average'), ('hierarchical', 'single')]
):
    cluster_metrics[_label] = clustering_metric_search(
        data,
        n_cluster_li = list(range(2, 16)),
        standardization = False,
        clustering_method = _params[0],
        linkage = _params[1])


# Metric results to DataFrame
cluster_metrics_df = (
    pd.concat(
        {k: v for k, v in cluster_metrics.items()},
        axis = 1
    )
    .swaplevel(0, 1, axis='columns')
    .sort_index(axis='columns')
)
# Save
# cluster_metrics_df.to_csv('_cluster_metrics.csv',
#     quoting=2)

Evaluating clustering metrics: 100%|██████████| 14/14 [00:49<00:00,  3.55s/it]
